# Cameo Extraction — Assigning a Specific Agent

Demonstrates two ways to pin a job to a specific agent (or agent pool) when submitting
an `@istari:extract` job against a Cameo `.mdzip` file using the `dassault_cameo` tool.

| Option | Approach | When to use |
|--------|----------|-------------|
| **1** | Drop to `platform.client.add_job()` directly | One-off, no library changes needed |
| **2** | Extend `JobDefinition` with agent fields | Repeated use, keeps the fluent style |

Because Cameo requires a licensed desktop installation on a specific machine, explicit
agent assignment is the expected pattern — you need the job to run on the machine where
Cameo is installed.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with the `@istari:dassault_cameo` module loaded and access to `@istari:extract`.
- The sample file `NCXTable-example.mdzip` in the same directory as this notebook.

### Credentials

Create a `.env` file next to this notebook:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

## 1 · Connect

In [1]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition, JobView

platform = IstariPlatform.from_env()

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

/Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/fluent/.venv/lib/python3.11/site-packages/istari_digital_client/log_utils.py:32: UserWarning: SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
  result = func(*args, **kwargs)
2026-05-18 18:43:02 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.


IstariPlatform connected to https://fileservice-v2.demo.istari.app


## 2 · Configure

In [ ]:
MDZIP_PATH      = Path.cwd() / "NCXTable-example.mdzip"
DISPLAY_NAME    = "NCXTable-example.mdzip"
EXTERNAL_ID     = "cameo-agent-assignment-demo"

# Set this to the agent ID you want to target (see Section 3 below).
# Leave as None to browse available agents first.
TARGET_AGENT_ID = "f380e804-f63b-451c-b272-45c728cf5556"   # e.g. "agt-abc123"

assert MDZIP_PATH.exists(), f"File not found: {MDZIP_PATH}"
print(f"Input file:  {MDZIP_PATH}")
print(f"Agent ID:    {TARGET_AGENT_ID or '(not set — browse agents in Section 3 first)'}")

Input file:  /Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/samples/NCXTable-example.mdzip
Agent ID:    (not set — browse agents in Section 3 first)


## 3 · Browse available agents

List all agents that have the `@istari:dassault_cameo` module loaded. Set `TARGET_AGENT_ID`
in the config cell above to the ID of the agent you want to use, then re-run from
Section 4 onwards.

In [3]:
agents = platform.client.list_agents(
    module_name="@istari:dassault_cameo",
    size=25,
)

print(f"Agents with @istari:dassault_cameo: {agents.total}\n")
print(f"{'AGENT ID':<40} {'DISPLAY NAME':<30} {'OS':<20} {'STATUS'}")
print("-" * 110)
for a in agents.items:
    display_name = a.display_name.display_name if a.display_name else ""
    status       = a.status.name if a.status else "Unknown"
    print(f"  {a.id:<38} {display_name:<30} {(a.host_os or ''):<20} {status}")

# Agents with status Idle are ready to accept work.
# Copy an ID above into TARGET_AGENT_ID in the config cell.

Agents with @istari:dassault_cameo: 1

AGENT ID                                 DISPLAY NAME                   OS                   STATUS
--------------------------------------------------------------------------------------------------------------
  f380e804-f63b-451c-b272-45c728cf5556   gentle-hurin-4683              Windows 10           AgentStatusName.IDLE


## 4 · Upload the Cameo model

In [4]:
model = platform.upload_model(
    MDZIP_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded model {model.id}")
print(model)

Uploaded model 36733c5b-3e5a-42d7-9990-1020dae32f94
Model('NCXTable-example.mdzip', id=36733c5b-3e5a-42d7-9990-1020dae32f94, file=fb306871-407e-44b6-b8c3-90b81f3a7a77, rev=bb662b49-57e7-40a8-aaff-7cea6f3c25f6)


## 5 · Option 1 — assign via `platform.client.add_job()`

Bypass `JobDefinition` entirely and call `add_job()` on the raw client.
This exposes every parameter the API supports, including `assigned_agent_id`
and `assigned_agent_pool_id`.

After submission, wrap the returned `Job` in a `JobView` to get `.wait()`,
`.get_products()`, and the rest of the fluent interface.

In [5]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

job1_raw = platform.client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name="dassault_cameo",
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",  # alternative: assign to a pool
)

job1 = JobView(_job=job1_raw, _client=platform.client)
print(f"Submitted job {job1.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job1.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 1 finished: {job1.status}")
print(f"Executed by:    {job1._job.agent_id or '(cloud)'}")

products1 = job1.get_products()
print(f"Products: {[p.name for p in products1]}")

AssertionError: Set TARGET_AGENT_ID in the config cell before running this section.

## 6 · Option 2 — extend `JobDefinition`

For repeated use, subclass `JobDefinition` to add the agent fields, then
provide a small helper that calls `add_job()` and returns a `JobView`.
This keeps the submission call site as clean as the standard fluent style
without patching the library.

> **Note:** Run this after Option 1 has fully completed and the agent has returned
> to `Idle`. After finishing a job the agent briefly enters `ExecutionSuccess` before
> cycling back — submitting a new assigned job during that window will leave it in
> `Pending` until the agent is ready again.

If you need this across multiple notebooks, move the class and helper into
a shared module (e.g. `istari_fluent/istari_utils.py`) and wire
`assigned_agent_id` through `_submit_job_impl`.

In [ ]:
from pydantic import Field as PydanticField


class AgentJobDefinition(JobDefinition):
    """JobDefinition extended with agent/pool targeting."""
    assigned_agent_id: str | None = PydanticField(default=None)
    assigned_agent_pool_id: str | None = PydanticField(default=None)


def submit_job(platform: IstariPlatform, model_id: str, defn: AgentJobDefinition) -> JobView:
    """Submit an AgentJobDefinition and return a JobView ready to .wait()."""
    job_raw = platform.client.add_job(
        model_id=model_id,
        function=defn.function,
        tool_name=defn.tool_name,
        tool_version=defn.tool_version,
        operating_system=defn.operating_system,
        parameters=defn.build_parameters(),
        assigned_agent_id=defn.assigned_agent_id,
        assigned_agent_pool_id=defn.assigned_agent_pool_id,
    )
    return JobView(_job=job_raw, _client=platform.client)

In [ ]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

extract_on_agent = AgentJobDefinition(
    function="@istari:extract",
    tool_name="dassault_cameo",
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",
)

job2 = submit_job(platform, model.id, extract_on_agent)
print(f"Submitted job {job2.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job2.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 2 finished: {job2.status}")
print(f"Executed by:    {job2._job.agent_id or '(cloud)'}")

products2 = job2.get_products()
print(f"Products: {[p.name for p in products2]}")

## Verify in the UI

1. **Jobs / Activity** — Both jobs should show `dassault_cameo / @istari:extract`.
2. **Job detail** — Each job should show the assigned agent under its execution details.

## Optional · Archive the model

In [ ]:
model.archive()